In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name","ecommerce","catalog_name")
dbutils.widgets.text("storage_account_name","abiecommerceadlsdev","storage_account_name")
dbutils.widgets.text("container_name","ecomm-raw-data","container_name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

print(catalog_name, storage_account_name, container_name)

In [0]:
silver_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoints/silver/fact_order_shipment"


In [0]:
df_bronze_shipment = spark.readStream.format("delta").table(f"{catalog_name}.bronze.brz_order_shipment")



In [0]:
df_silver_shipment = df_bronze_shipment.withColumn("order_dt", F.col("order_dt").cast("date"))
df_silver_shipment = df_silver_shipment.withColumn("carrier", F.trim(F.upper(F.col("carrier"))))
df_silver_shipment = df_silver_shipment.withColumn("processed_time", F.current_timestamp())

In [0]:
def upsert_to_silver(microBatchDF, batchId):
    table_name = f"{catalog_name}.silver.slv_order_shipment"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("silver_table").merge(
            microBatchDF.alias("batch_table"),
            "silver_table.shipment_id = batch_table.shipment_id" ,
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()    


In [0]:
df_silver_shipment.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_silver
).format("delta").option("checkpointLocation", silver_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()